# What a plan can and cannot tell you

> How far to trust trufl

::: {.callout-note appearance="simple" icon=false}
**Explanation: understand why.** Read it away from the work. There is nothing to run. Want to act? Use a [how-to guide](../how-to/02_use_your_data.ipynb). [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

trufl is alpha software and has not been validated for operational use. Beyond that general caution, it helps to know what kind of claim a plan makes: *given these data and these priorities, these are reasonable places to look next*. Several stronger claims are easy to read into it.

## It does not say where it is safe

trufl has no notion of dose, exclusion zones or access, either for the public or for the team it sends. A low priority can mean clean, and can equally mean contaminated and already well measured.

## It does not estimate contamination

The state summarises samples. It is not an estimate of the mean of an area, has no confidence interval and takes priors and measurements as exact. Samples placed by priority are also unevenly spread by design, so a simple average over all of them can overstate the contamination of the whole region.

## It has not been shown to beat chance

It is plausible that following priorities is better than sampling at random, but these pages contain no evidence of it. trufl cannot yet [score a campaign](https://github.com/franckalbinet/trufl/issues/12) or [compare strategies](https://github.com/franckalbinet/trufl/issues/13). The examples also flatter it: their prior comes from the same map as their measurements, so the survey is never wrong.

## Moran's I finds trends that are not there

A criterion can be reasonable in theory and misleading in practice. Moran's I should show whether an area has a spatial trend. One area rarely has enough points, so trufl borrows the nearest measurements of each point, and in doing so counts the same measurement several times. A point and its own copy agree perfectly, which looks exactly like a trend.

If every area contains nothing but random noise, a sound test reports a trend in about 5% of them. trufl reports:

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

from trufl.callbacks import MoranICB
areas = gridder('../files/ground-truth-01-4326-simulated.tif')
found = {}
for per_area in (6, 12):
    rng = np.random.default_rng(0)
    noise = Sampler(areas).sample(np.full(len(areas), per_area), method='uniform', rng=rng).explode(index_parts=False)
    noise['value'] = rng.random(len(noise))
    np.random.seed(0)
    found[per_area] = State(noise, areas, [MoranICB()])()['Moran.I'].notna().mean()
test_eq(min(found.values()) > 0.5, True)

In [ ]:
#| echo: false
pd.DataFrame({'areas with a reported trend': [f'{v:.0%}' for v in found.values()]},
             index=pd.Index(list(found), name='samples per area'))

,areas with a reported trend
samples per area,
6,72%
12,100%


For this reason none of the examples in this documentation ranks on Moran's I.

## What it is good for

Within those bounds a plan is useful for what it makes visible. It shows which data were used, which priorities were chosen and what follows from them. A group can rehearse with it before an emergency, argue about the weights while there is time, and during one explain why a team went to one village and not the next.